In [ ]:
# Supress warnings
import warnings
warnings.filterwarnings('ignore')

#Import libraies 
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import re
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation, NMF
import gensim
from gensim.models import CoherenceModel
import gensim.corpora as corpora
nltk.download('stopwords')
nltk.download('wordnet')
#Load the dataset
import pandas as pd
data = pd.read_csv('D:/Project/rows.csv')

#See head
data.head()

In [ ]:
# Drop nulls and sample 10k randomly
complaints = data['Consumer complaint narrative'].dropna().sample(10000, random_state=42).tolist()

In [3]:
# Preprocessing
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

In [4]:
def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)  # Strip punctuation
    tokens = text.split()  # Tokenize
    tokens = [token for token in tokens if token not in stop_words]  # Remove stopwords
    tokens = [lemmatizer.lemmatize(token) for token in tokens]  # Lemmatize
    tokens = [token for token in tokens if token.isalpha() and len(token) >= 3]  # Alphabetic only, min length 3
    return ' '.join(tokens)

processed_complaints = [preprocess(complaint) for complaint in complaints]


In [5]:
# Vectorization
# Bag-of-Words
count_vectorizer = CountVectorizer(max_df=0.9, min_df=10)
bow_matrix = count_vectorizer.fit_transform(processed_complaints)

# TF-IDF
tfidf_vectorizer = TfidfVectorizer(max_df=0.9, min_df=10)
tfidf_matrix = tfidf_vectorizer.fit_transform(processed_complaints)


In [6]:
# Topic Extraction with Coherence Optimization
# Prepare Gensim
texts = [doc.split() for doc in processed_complaints]
id2word = corpora.Dictionary(texts)
corpus = [id2word.doc2bow(text) for text in texts]

In [7]:
def compute_coherence(model_type, vectorizer, matrix, n_topics):
    if model_type == 'LDA':
        model = LatentDirichletAllocation(n_components=n_topics, random_state=42)
        model.fit(matrix)
    elif model_type == 'NMF':
        model = NMF(n_components=n_topics, random_state=42)
        model.fit(matrix)
    
    # Get topics for coherence
    topics = []
    feature_names = vectorizer.get_feature_names_out()
    for topic_idx, topic in enumerate(model.components_):
        topics.append([feature_names[i] for i in topic.argsort()[:-10 - 1:-1]])
    
    coherence_model = CoherenceModel(topics=topics, texts=texts, dictionary=id2word, coherence='c_v')
    return coherence_model.get_coherence(), model

# Optimize number of topics (3 to 10)
best_score = 0
best_model = None
best_n = 0
best_type = ''
best_vectorizer = None

In [8]:
for n in range(3, 11):
    for model_type in ['LDA', 'NMF']:
        for vec_type, vec, matrix in [('BoW', count_vectorizer, bow_matrix), ('TF-IDF', tfidf_vectorizer, tfidf_matrix)]:
            score, model = compute_coherence(model_type, vec, matrix, n)
            print(f'{model_type} with {vec_type}, {n} topics: Coherence = {score}')
            if score > best_score:
                best_score = score
                best_model = model
                best_n = n
                best_type = model_type
                best_vectorizer = vec

# Display top words for best model
feature_names = best_vectorizer.get_feature_names_out()
for topic_idx, topic in enumerate(best_model.components_):
    print(f"Topic {topic_idx + 1}:")
    print(" ".join([feature_names[i] for i in topic.argsort()[:-10 - 1:-1]]))

# Save results
pd.DataFrame({'Processed Complaints': processed_complaints}).to_csv('D:/Project/Consumer-Complaint-NLP/processed_data.csv', index=False)

LDA with BoW, 3 topics: Coherence = 0.44326503479791785
LDA with TF-IDF, 3 topics: Coherence = 0.4610537867500432
NMF with BoW, 3 topics: Coherence = 0.4844727367903546
NMF with TF-IDF, 3 topics: Coherence = 0.4988585914731644
LDA with BoW, 4 topics: Coherence = 0.46374219419793483
LDA with TF-IDF, 4 topics: Coherence = 0.48622291287993114
NMF with BoW, 4 topics: Coherence = 0.4709136123596256
NMF with TF-IDF, 4 topics: Coherence = 0.5199606211225307
LDA with BoW, 5 topics: Coherence = 0.4694695978463013
LDA with TF-IDF, 5 topics: Coherence = 0.46068032947998694
NMF with BoW, 5 topics: Coherence = 0.49663609978853396
NMF with TF-IDF, 5 topics: Coherence = 0.5342878163034068
LDA with BoW, 6 topics: Coherence = 0.49131428740585076
LDA with TF-IDF, 6 topics: Coherence = 0.46838885124052254
NMF with BoW, 6 topics: Coherence = 0.5106605970658792
NMF with TF-IDF, 6 topics: Coherence = 0.5165546850508328
LDA with BoW, 7 topics: Coherence = 0.4763965696922422
LDA with TF-IDF, 7 topics: Coheren